# Flux LoRA Training — Colab T4

Тренировка кастомной LoRA конкретного человека с помощью **ai-toolkit** (Ostris) на бесплатном T4 GPU.

**Что вы получите:** LoRA файл (.safetensors), который можно использовать в ComfyUI с нодой LoraLoader для генерации изображений человека в любом стиле/окружении.

**Время:** ~40-70 мин (5-10 мин квантизация + кэширование, затем 30-60 мин тренировка на GPU)
**VRAM:** ~10-14 GB на этапе тренировки. Модель квантизуется поблочно на GPU (qfloat8).

---

### Как это работает (пошагово)

```
[1. GPU] → [2. HF Токен] → [3. Установка] → [4. ЗАГРУЗКА ФОТО] → [5. Тренировка] → [6. Тест] → [7. Скачать]
                                                     ↑
                                             Самый важный шаг!
                                        Качество фото = качество LoRA
```

### Фазы тренировки (ячейка 5)

| Фаза | Время | GPU VRAM | Что происходит |
|------|-------|----------|----------------|
| **1. Квантизация** | ~3-5 мин | Средний | Модель загружается поблочно на GPU и сжимается (24 ГБ → ~12 ГБ) |
| **2. Кэширование** | ~1-3 мин | Средний | Изображения кодируются в латенты, текст — в эмбеддинги |
| **3. Тренировка** | ~30-50 мин | **10-14 ГБ** | LoRA обучается на GPU |

> **Важно:** Swap-файл подключается в ячейке 3 как страховка от крэшей по RAM.

---

### Сколько фото нужно?

| Количество | Рекомендация | Комментарий |
|:----------:|:------------:|-------------|
| **< 10** | Мало | Модель не выучит лицо — слишком мало данных |
| **10-20** | Минимум | Работает, если фото разнообразные и качественные |
| **20-40** | Идеально | Лучший баланс качества и скорости тренировки |
| **40-70** | Хорошо | Больше данных, но нужно следить за шагами тренировки |
| **70-100+** | Можно | Работает, но избыток похожих фото ведёт к переобучению |

> **Правило:** шаги подбираются автоматически в ячейке 5 (~40 шагов на фото). Чем больше фото — тем больше шагов, но дольше тренировка.

### Идеальный датасет — чек-лист

Главный принцип: **лицо одинаковое, всё остальное разное**.

#### Ракурсы (обязательно все 3 типа):

| Тип | Сколько | Что это |
|-----|:-------:|---------|
| **Анфас** (прямо в камеру) | 30-40% | Основа для узнаваемости лица |
| **3/4** (полуповорот) | 30-40% | Самый частый ракурс в генерации |
| **Профиль** (сбоку) | 20-30% | Модель учит форму носа, подбородка, лба |

#### Кадрирование (чем разнообразнее, тем лучше):

| Тип | Сколько | Что на фото |
|-----|:-------:|-------------|
| **Крупный план** (лицо) | ~30% | Только голова и плечи |
| **Портрет** (по грудь/пояс) | ~40% | Голова + верхняя часть тела |
| **В полный рост** | ~30% | Всё тело, лицо занимает меньшую часть |

#### Разнообразие (ключ к качеству):

| Параметр | Нужно разнообразие | Зачем |
|----------|:------------------:|-------|
| **Освещение** | Улица, помещение, студия, вечер | Модель не привяжется к одному свету |
| **Фон** | Разные места и цвета | Модель учит лицо, а не обои |
| **Одежда** | Разная (повседневная, нарядная) | Модель не "запомнит" конкретную одежду |
| **Выражения** | Улыбка, серьёзное, смех, задумчивость | Естественные результаты при генерации |

### Чего НЕ нужно

| Плохо | Почему |
|-------|--------|
| Групповые фото (2+ человека) | Модель не поймёт, кого учить |
| Тёмные очки / маска на лице | Лицо скрыто — модель не выучит черты |
| Сильный фильтр / обработка | Модель выучит фильтр, а не лицо |
| 50 селфи с одного ракурса | Переобучение — модель запомнит позу и фон |
| Размытые / тёмные фото | Мусор на входе = мусор на выходе |
| Фото другого человека в датасете | Модель смешает два лица |

### Переобучение vs Недообучение

| Проблема | Симптомы | Решение |
|----------|----------|---------|
| **Переобучение** | Все фото выглядят одинаково, копируется фон из датасета | Больше разнообразия в фото, меньше шагов |
| **Недообучение** | Лицо не узнаётся, генерация игнорирует триггер-слово | Больше шагов, лучше качество фото, больше фото |

**Запускайте ячейки 1-7 по порядку.**

In [ ]:
#@title 1. Проверка GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
#@title 2. Авторизация Hugging Face (обязательно для FLUX)
#@markdown ---
#@markdown ### Зачем это нужно?
#@markdown Модель **FLUX.1-dev** от Black Forest Labs — **закрытая модель** на Hugging Face.
#@markdown Для скачивания нужно:
#@markdown 1. Зарегистрироваться на [huggingface.co](https://huggingface.co)
#@markdown 2. Принять лицензию на странице модели: [FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev)
#@markdown 3. Создать токен: [Settings → Access Tokens](https://huggingface.co/settings/tokens) (тип: **Read**)
#@markdown 4. Вставить токен в защищённое поле ниже

import getpass

print("Введите ваш Hugging Face токен (начинается с hf_...):")
print("(токен не будет отображаться при вводе)\n")
hf_token = getpass.getpass("HF Token: ")

if not hf_token or not hf_token.strip().startswith("hf_"):
    print("=" * 60)
    print("  ОШИБКА: Токен не указан или неверный формат!")
    print()
    print("  1. Перейдите: https://huggingface.co/settings/tokens")
    print("  2. Создайте токен с доступом Read")
    print("  3. Вставьте его в поле выше")
    print("  4. Перезапустите эту ячейку")
    print("=" * 60)
    raise ValueError("Hugging Face токен обязателен для FLUX.1-dev")

!pip install huggingface_hub -q
from huggingface_hub import login
login(token=hf_token.strip())

import os
os.environ["HF_TOKEN"] = hf_token.strip()

print("\n" + "=" * 60)
print("  Авторизация успешна!")
print("  Теперь можно скачивать FLUX.1-dev")
print("=" * 60)

In [ ]:
#@title 3. Установка ai-toolkit + Зависимости
#@markdown ---
#@markdown Эта ячейка также подключает **swap-файл 8 ГБ** — страховку от крэшей по RAM
#@markdown при загрузке тяжёлых моделей. Colab free даёт ~12.7 ГБ RAM, а FLUX требует больше.

# ── Swap-файл (страховка от OOM по системной RAM) ────────────────────
import os
if not os.path.exists("/content/swapfile"):
    print("Подключение swap 8 ГБ (страховка от крэшей по RAM)...")
    !fallocate -l 8G /content/swapfile && chmod 600 /content/swapfile && mkswap /content/swapfile && swapon /content/swapfile
    !free -h | head -3
    print("Swap подключён!\n")
else:
    !swapon /content/swapfile 2>/dev/null; true
    print("Swap уже подключён\n")

# ── ai-toolkit ───────────────────────────────────────────────────────
%cd /content
!test -d ai-toolkit || git clone https://github.com/ostris/ai-toolkit.git
%cd /content/ai-toolkit
!git submodule update --init --recursive

# Обновляем pip/setuptools/wheel
!pip install --upgrade pip setuptools wheel -q

# Фиксируем numpy < 2.0 (ai-toolkit несовместим с numpy 2.x)
!echo "numpy<2.0" > /tmp/numpy_constraint.txt
!pip install "numpy>=1.26,<2.0" -q

# Ставим зависимости с ограничением numpy
!pip install -r requirements.txt -q -c /tmp/numpy_constraint.txt
!pip install peft accelerate safetensors PyYAML bitsandbytes -q -c /tmp/numpy_constraint.txt
!pip install diffusers transformers -q -c /tmp/numpy_constraint.txt
!pip install psutil -q

# Проверяем numpy < 2.0
import numpy as np
assert int(np.__version__.split(".")[0]) < 2, f"numpy {np.__version__} >= 2.0! Перезапустите runtime и ячейку."
print(f"\nnumpy версия: {np.__version__}")
print("ai-toolkit установлен!")

In [ ]:
#@title 4. Загрузка тренировочных фото
#@markdown ## Нажмите кнопку "Выбрать файлы" ниже
#@markdown
#@markdown Загрузите фотографии человека, на которого будете тренировать LoRA.
#@markdown
#@markdown **Рекомендуемое количество:** 20-40 фото (можно больше — шаги подстроятся автоматически).
#@markdown
#@markdown Фото сохранятся в папку `/content/dataset/`.
#@markdown
#@markdown **Форматы:** .jpg, .jpeg, .png, .webp
#@markdown
#@markdown ---
#@markdown
#@markdown ### Чек-лист (подробности в ячейке 0):
#@markdown - На каждом фото **один человек**
#@markdown - Лицо **хорошо видно** (без очков, масок)
#@markdown - Есть фото **анфас + 3/4 + профиль**
#@markdown - Есть **крупный план + портрет + полный рост**
#@markdown - Разное освещение, фон, одежда, выражения
#@markdown - Фото **чёткие**, без фильтров

from google.colab import files
from IPython.display import display, Image as IPImage
import os

DATASET_DIR = "/content/dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

print("=" * 60)
print("  ЗАГРУЗКА ФОТО ДЛЯ ТРЕНИРОВКИ LoRA")
print("=" * 60)
print(f"\n  Папка датасета: {DATASET_DIR}")
print("  Рекомендуется: 20-40 фото (можно больше)\n")

uploaded = files.upload()

VALID_EXT = ('.jpg', '.jpeg', '.png', '.webp')
saved = 0
for name, data in uploaded.items():
    if not name.lower().endswith(VALID_EXT):
        print(f"  Пропущено (неподдерживаемый формат): {name}")
        continue
    path = os.path.join(DATASET_DIR, name)
    with open(path, "wb") as f:
        f.write(data)
    saved += 1

# Показываем итог
all_files = [f for f in os.listdir(DATASET_DIR) if f.lower().endswith(VALID_EXT)]
count = len(all_files)

print(f"\n{'=' * 60}")
print(f"  Загружено: {saved} фото")
print(f"  Всего в датасете: {count} фото")
print(f"  Папка: {DATASET_DIR}")
print(f"{'=' * 60}")

if count < 10:
    print(f"\n  МАЛО: {count} фото — модель может не выучить лицо.")
    print("  Рекомендуется минимум 15, идеально 20-40.")
elif count <= 40:
    print(f"\n  Отлично! {count} фото — идеальный размер датасета.")
elif count <= 70:
    print(f"\n  Хорошо! {count} фото — большой датасет.")
    print("  Шаги тренировки подстроятся автоматически.")
else:
    print(f"\n  {count} фото — очень большой датасет.")
    print("  Убедитесь, что фото разнообразные (не 100 селфи с одного ракурса).")
    print("  Шаги тренировки подстроятся автоматически.")

# Авторасчёт шагов для информации
rec_steps = max(800, min(4000, count * 40))
print(f"\n  Рекомендуемые шаги для {count} фото: ~{rec_steps}")
print(f"  (будет подставлено автоматически в ячейке 5, если steps=0)")

# Превью загруженных фото
print(f"\nПревью (первые 8 фото):")
for f in sorted(all_files)[:8]:
    path = os.path.join(DATASET_DIR, f)
    print(f"  {f}")
    try:
        display(IPImage(filename=path, width=200))
    except Exception:
        pass

In [ ]:
#@title 5. Настройка и запуск тренировки
#@markdown Задайте параметры тренировки ниже.

trigger_word = "ohwx" #@param {type:"string"}
#@markdown Триггер-слово — уникальный токен, который активирует LoRA. Используйте что-то редкое, например "ohwx" или "sks".

steps = 0 #@param {type:"integer"}
#@markdown Количество шагов. **0 = автоподбор** (~40 шагов на фото, диапазон 800-4000). Или задайте вручную.

learning_rate = 1e-4 #@param {type:"number"}
resolution = 512 #@param [512, 768] {type:"raw"}
#@markdown Разрешение тренировочных изображений. **512** — безопасный выбор для T4. **768** — лучше качество, но рискованнее по VRAM.
lora_rank = 8 #@param [4, 8, 16] {type:"raw"}
#@markdown LoRA rank — влияет на ёмкость модели. **8** — безопасный выбор для T4. **16** — лучше качество, но больше VRAM.
output_name = "my_person_lora" #@param {type:"string"}

import yaml, os, gc, torch, subprocess, threading, time

# --- Проверка датасета ---
DATASET_DIR = "/content/dataset"
VALID_EXT = ('.jpg', '.jpeg', '.png', '.webp')
dataset_files = [f for f in os.listdir(DATASET_DIR) if f.lower().endswith(VALID_EXT)] if os.path.isdir(DATASET_DIR) else []
num_images = len(dataset_files)

if num_images == 0:
    print("=" * 60)
    print("  ОШИБКА: В папке /content/dataset/ нет фото!")
    print("  Запустите ячейку 4 (Загрузка фото) сначала.")
    print("=" * 60)
    raise FileNotFoundError("Датасет пуст — загрузите фото в ячейке 4")

print(f"Фото в датасете: {num_images}")

# --- Автоподбор шагов ---
if steps == 0:
    steps = max(800, min(4000, num_images * 40))
    print(f"Автоподбор шагов: {num_images} фото x ~40 = {steps} шагов")
else:
    ratio = steps / max(num_images, 1)
    print(f"Ручной выбор: {steps} шагов ({ratio:.0f} шагов/фото)")
    if ratio > 80:
        print(f"  ВНИМАНИЕ: {ratio:.0f} шагов/фото — высокий риск переобучения.")
        print(f"  Рекомендуется: ~40 шагов/фото = {num_images * 40} шагов.")
    elif ratio < 15:
        print(f"  ВНИМАНИЕ: {ratio:.0f} шагов/фото — модель может недообучиться.")
        print(f"  Рекомендуется: ~40 шагов/фото = {num_images * 40} шагов.")

# --- Проверка HF токена ---
hf_token = os.environ.get("HF_TOKEN", "")
if not hf_token:
    print("=" * 60)
    print("  ОШИБКА: HF_TOKEN не найден!")
    print("  Запустите ячейку 2 (Авторизация HF) заново.")
    print("=" * 60)
    raise ValueError("HF_TOKEN не установлен — запустите ячейку 2")

# Сохраняем токен на диск, чтобы subprocess ai-toolkit его подхватил
from huggingface_hub import login
login(token=hf_token)
print("HF токен активен и сохранён на диск.")

# --- Проверка GPU ---
if not torch.cuda.is_available():
    print("CUDA не доступна! Тренировка на GPU невозможна.")
    raise RuntimeError("CUDA не найдена — проверьте, что Runtime → GPU включён")

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
vram_total = torch.cuda.get_device_properties(0).total_memory
vram_gb = vram_total / 1024**3
gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}, VRAM: {vram_gb:.1f} GiB")

if resolution == 768 and vram_gb < 14.0:
    print(f"\n  ВНИМАНИЕ: Разрешение 768 при {vram_gb:.1f} ГБ VRAM — рискованно.")
    print(f"  Если возникнет OOM, переключитесь на 512 и rank 4.\n")

# Борьба с фрагментацией памяти CUDA
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Проверка swap ---
swap_ok = os.path.exists("/content/swapfile")
if not swap_ok:
    print("\n  ВНИМАНИЕ: Swap не подключён! Запустите ячейку 3 заново.")
    print("  Без swap runtime может крашнуться при загрузке модели.\n")

# --- Агрессивная очистка RAM перед тренировкой ---
import ctypes
gc.collect()
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass

# --- Проверка доступной RAM ---
try:
    import psutil
    ram = psutil.virtual_memory()
    swap = psutil.swap_memory()
    print(f"RAM: {ram.available / 1024**3:.1f} / {ram.total / 1024**3:.1f} ГБ свободно")
    print(f"Swap: {swap.free / 1024**3:.1f} / {swap.total / 1024**3:.1f} ГБ свободно")
except ImportError:
    pass

# --- Конфиг тренировки ---
# Архитектура памяти для T4 (16 ГБ VRAM, ~12.7 ГБ RAM + 8 ГБ swap):
#
# КРИТИЧНО: НЕ используем low_vram: True!
# low_vram загружает ВСЮ модель (~24 ГБ fp16) в системную RAM для
# квантизации на CPU. Colab free = ~12.7 ГБ RAM → OOM → runtime убит.
#
# Без low_vram квантизация идёт ПОБЛОЧНО на GPU:
# каждый блок трансформера загружается → квантизуется → сохраняется.
# Пиковое потребление CPU RAM минимально.
#
# НЕ используем layer_offloading — есть баг с квантизованными тензорами
# (ai-toolkit issue #510: torchao .to() incompatible with quantized tensors).
#
# dtype: fp16 — ОБЯЗАТЕЛЬНО для T4 (Turing). T4 НЕ поддерживает bf16.

# Интервал сэмплирования — каждые ~25% тренировки, но не реже чем раз в 500 шагов
sample_every = max(250, min(500, steps // 4))

config = {
    "job": "extension",
    "config": {
        "name": output_name,
        "process": [{
            "type": "sd_trainer",
            "training_folder": "/content/output",
            "performance_log_every": 100,
            "device": "cuda:0",
            "trigger_word": trigger_word,
            "network": {
                "type": "lora",
                "linear": lora_rank,
                "linear_alpha": lora_rank
            },
            "save": {
                "dtype": "float16",
                "save_every": sample_every,
                "max_step_saves_to_keep": 2
            },
            "datasets": [{
                "folder_path": DATASET_DIR,
                "caption_ext": "txt",
                "caption_dropout_rate": 0.05,
                "shuffle_tokens": False,
                "cache_latents_to_disk": True,
                "resolution": [resolution, resolution]
            }],
            "train": {
                "batch_size": 1,
                "steps": steps,
                "gradient_accumulation_steps": 1,
                "train_unet": True,
                "train_text_encoder": False,
                "gradient_checkpointing": True,
                "noise_scheduler": "flowmatch",
                "optimizer": "adamw8bit",
                "lr": learning_rate,
                "ema_config": {"use_ema": False},
                "dtype": "fp16"
            },
            "model": {
                "name_or_path": "black-forest-labs/FLUX.1-dev",
                "is_flux": True,
                "quantize": True,
                "quantize_te": True
            },
            "sample": {
                "sampler": "flowmatch",
                "sample_every": sample_every,
                "width": 512,
                "height": 512,
                "prompts": [
                    f"a professional photo of {trigger_word} person, studio lighting, 4k"
                ],
                "neg": "",
                "seed": 42,
                "walk_seed": True,
                "guidance_scale": 3.5,
                "sample_steps": 20
            }
        }]
    }
}

config_path = "/content/ai-toolkit/config/train_lora.yaml"
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"\nКонфиг сохранён: {config_path}")
print(f"\nПараметры тренировки:")
print(f"  Датасет: {num_images} фото")
print(f"  Триггер-слово: {trigger_word}")
print(f"  Шаги: {steps} (~{steps / max(num_images, 1):.0f} на фото)")
print(f"  Скорость обучения: {learning_rate}")
print(f"  Разрешение: {resolution}x{resolution}")
print(f"  LoRA rank: {lora_rank}")
print(f"  dtype: fp16 (T4 не поддерживает bf16)")
print(f"  Квантизация: qfloat8 поблочно на GPU")
print(f"  Квантизация text encoders: включена")
print(f"  Gradient checkpointing: включён")
print(f"  Optimizer: AdamW 8-bit")
print(f"  Сэмплирование: каждые {sample_every} шагов (512x512)")

print(f"\n{'='*60}")
print(f"  ФАЗЫ ТРЕНИРОВКИ:")
print(f"  1. Загрузка + квантизация на GPU (~3-5 мин)")
print(f"     -> Модель загружается поблочно, не забивает RAM")
print(f"  2. Кэширование латентов + эмбеддингов (~1-3 мин)")
print(f"  3. Тренировка LoRA на GPU (~30-60 мин)")
print(f"     -> VRAM: ~10-14 ГБ")
print(f"{'='*60}")

# --- Фоновый мониторинг GPU VRAM + RAM ---
vram_log_path = "/content/vram_monitor.log"
stop_monitor = threading.Event()

def monitor_resources():
    """Логирует VRAM и RAM каждые 30 сек в файл для диагностики."""
    with open(vram_log_path, "w") as log:
        while not stop_monitor.is_set():
            try:
                # GPU VRAM
                result = subprocess.run(
                    ["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
                     "--format=csv,noheader,nounits"],
                    capture_output=True, text=True, timeout=5
                )
                vram_info = ""
                if result.returncode == 0:
                    parts = result.stdout.strip().split(", ")
                    if len(parts) == 3:
                        vram_info = f"VRAM: {parts[0]}/{parts[1]} MiB ({parts[2]}% GPU)"

                # System RAM
                ram_info = ""
                try:
                    import psutil
                    ram = psutil.virtual_memory()
                    swap = psutil.swap_memory()
                    ram_info = f"RAM: {ram.used // 1024**2}/{ram.total // 1024**2} MiB | Swap: {swap.used // 1024**2}/{swap.total // 1024**2} MiB"
                except ImportError:
                    pass

                if vram_info or ram_info:
                    ts = time.strftime("%H:%M:%S")
                    line = f"[{ts}] {vram_info} | {ram_info}"
                    log.write(line + "\n")
                    log.flush()
            except Exception:
                pass
            stop_monitor.wait(30)

monitor_thread = threading.Thread(target=monitor_resources, daemon=True)
monitor_thread.start()
print(f"\nМониторинг GPU + RAM запущен (лог: {vram_log_path})")

print(f"\nЗапуск тренировки...\n")

# --- Запуск тренировки ---
!cd /content/ai-toolkit && \
    CUDA_VISIBLE_DEVICES=0 \
    PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
    python run.py config/train_lora.yaml

# --- Останавливаем мониторинг ---
stop_monitor.set()
monitor_thread.join(timeout=5)

# --- Итоговый отчёт по ресурсам ---
print(f"\n{'='*60}")
print(f"  ИТОГИ ТРЕНИРОВКИ — ИСПОЛЬЗОВАНИЕ РЕСУРСОВ")
print(f"{'='*60}")

# Показываем лог
if os.path.exists(vram_log_path):
    with open(vram_log_path) as f:
        lines = f.readlines()
    if lines:
        print(f"\n  Записей в логе: {len(lines)}")
        max_vram = 0
        max_line = ""
        for line in lines:
            try:
                used = int(line.split("VRAM: ")[1].split("/")[0])
                if used > max_vram:
                    max_vram = used
                    max_line = line.strip()
            except (IndexError, ValueError):
                pass
        if max_vram > 0:
            print(f"  Пиковое VRAM: {max_vram} MiB ({max_vram/1024:.1f} GiB)")
            print(f"  {max_line}")

        print(f"\n  Начало:")
        for line in lines[:3]:
            print(f"    {line.strip()}")
        if len(lines) > 6:
            print(f"  ...")
        print(f"  Конец:")
        for line in lines[-3:]:
            print(f"    {line.strip()}")
    else:
        print("  Лог пуст — nvidia-smi не вернула данных")

# Текущее состояние GPU
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

print(f"{'='*60}")
print(f"\nТренировка завершена!")

In [ ]:
#@title 6. Тест LoRA — Генерация тестового фото
#@markdown Быстрый тест обученной LoRA через пайплайн diffusers.
#@markdown
#@markdown Модель загружается в 4-bit квантизации (NF4) с автоматическим распределением по устройствам.

trigger_word = "ohwx" #@param {type:"string"}
prompt = "a photo of ohwx person, professional headshot, studio lighting, 4k" #@param {type:"string"}
output_name = "my_person_lora" #@param {type:"string"}

import glob, os, gc, torch

# Находим последний LoRA файл
lora_files = sorted(glob.glob(f"/content/output/{output_name}/*.safetensors"))
if not lora_files:
    print("LoRA файл не найден! Убедитесь, что тренировка завершена.")
else:
    lora_path = lora_files[-1]
    print(f"Используется LoRA: {lora_path}")

    # Очистка VRAM перед загрузкой pipeline
    gc.collect()
    torch.cuda.empty_cache()

    from diffusers import FluxPipeline
    from transformers import BitsAndBytesConfig

    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        print("HF_TOKEN не найден! Запустите ячейку 2 (Авторизация HF) заново.")
        raise ValueError("HF_TOKEN не установлен")

    # NF4 квантизация — модель занимает ~6 ГБ вместо ~24 ГБ
    nf4_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    # device_map="auto" нужен для NF4 — распределяет слои между GPU и CPU.
    # НЕ совместим с enable_sequential_cpu_offload() (конфликт управления устройствами).
    pipe = FluxPipeline.from_pretrained(
        "black-forest-labs/FLUX.1-dev",
        torch_dtype=torch.float16,
        quantization_config=nf4_config,
        device_map="auto",
        token=hf_token
    )
    pipe.load_lora_weights(lora_path)

    image = pipe(
        prompt,
        num_inference_steps=25,
        guidance_scale=3.5,
        generator=torch.Generator("cuda").manual_seed(42)
    ).images[0]

    image.save("/content/test_result.png")

    from IPython.display import display, Image
    display(Image("/content/test_result.png"))
    print("Тестовое изображение сохранено: /content/test_result.png")

    del pipe
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
#@title 7. Скачивание LoRA / Сохранение на Google Drive
#@markdown Выберите, куда сохранить обученный LoRA файл.

save_to_drive = True #@param {type:"boolean"}
output_name = "my_person_lora" #@param {type:"string"}

import glob, shutil, os

lora_files = sorted(glob.glob(f"/content/output/{output_name}/*.safetensors"))
if not lora_files:
    print("LoRA файл не найден!")
else:
    lora_path = lora_files[-1]
    lora_filename = os.path.basename(lora_path)
    print(f"LoRA файл: {lora_path} ({os.path.getsize(lora_path) / 1024**2:.1f} MB)")

    if save_to_drive:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_dir = "/content/drive/MyDrive/ComfyUI_LoRAs"
        os.makedirs(drive_dir, exist_ok=True)
        dest = f"{drive_dir}/{lora_filename}"
        shutil.copy2(lora_path, dest)
        print(f"Сохранено на Drive: {dest}")
    else:
        from google.colab import files
        files.download(lora_path)
        print("Скачивание начато!")

---
## Использование LoRA в ComfyUI

### Способ 1: Нода LoraLoader
1. Скопируйте LoRA файл в `ComfyUI/models/loras/`
2. В вашем воркфлоу добавьте ноду **LoraLoader**
3. Подключите её между загрузчиком модели и KSampler
4. Выберите ваш LoRA файл
5. Используйте триггер-слово в промпте (например, "a photo of ohwx person in a garden")

### Способ 2: В Colab
Используйте ячейку "Скачать LoRA" в `colab_flux_photo.ipynb` для прямой загрузки.

### Советы
- **Триггер-слово** должно присутствовать в промпте для активации LoRA
- **Сила (strength)** 0.7-1.0 лучше всего работает для LoRA лиц
- **Пониженная сила** (0.4-0.6) для большего стилистического разнообразия
- Комбинируйте с IPAdapter FaceID для ещё лучшей консистентности

### Решение проблем

| Проблема | Причина | Решение |
|----------|---------|---------|
| **Runtime крашится (killed)** | Нехватка системной RAM | Убедитесь, что swap подключён (ячейка 3). НЕ включайте `low_vram: True` — он грузит 24 ГБ в RAM |
| **CUDA OOM** | Нехватка GPU VRAM | Уменьшите rank до 4, resolution до 512 |
| **Переобучение** | Все фото одинаковые, копируется фон | Больше разнообразия в фото, меньше шагов |
| **Недообучение** | Лицо не распознаётся | Больше шагов, лучше качество фото |
| **Медленная тренировка** | GPU не подключён | Runtime → Change runtime type → T4 GPU |
| **HF ошибка доступа** | Токен / лицензия | Примите лицензию FLUX.1-dev на HF, пересоздайте токен |

### Архитектура памяти (для понимания)

```
┌─────────────────────────────────────────────────┐
│                  Colab T4                        │
│                                                  │
│  CPU RAM (~12.7 ГБ + 8 ГБ swap)                │
│  ├── Python + PyTorch overhead (~2-3 ГБ)        │
│  ├── Датасет в памяти (минимальный)             │
│  └── Буферы загрузки (~1-2 ГБ)                 │
│                                                  │
│  GPU VRAM (16 ГБ)                               │
│  ├── Квантизованный трансформер (~10-12 ГБ)     │
│  ├── LoRA адаптер (rank 8 = ~100 МБ)           │
│  ├── Optimizer AdamW8bit (~200 МБ)              │
│  └── Градиенты + активации (~1-2 ГБ)           │
│       (gradient checkpointing экономит ~30%)     │
└─────────────────────────────────────────────────┘
```